# Annotation Union Merge + Disagreement Analysis

This notebook merges Reva and Ryan multi-label annotations:

1. Parse labels with blank/NaN → `_unknown`
2. Take the union of label sets
3. Remove `_unknown` when substantive labels exist
4. Serialize output labels deterministically

It then enriches `merged_glossary.tsv` with `source_domain` and `tier` columns from the provenance-classified glossary, and runs a disagreement analysis on both labeling tasks to assess whether the naive union-merge scheme is appropriate.

Writes:
- `merged_labels.tsv`
- `merged_glossary.tsv` (with `source_domain`, `tier`)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Resolve annotations directory whether notebook is run from annotations/ or repo root.
CWD = Path.cwd()
ANNOTATIONS_DIR = CWD if (CWD / "reva_labels.tsv").exists() else CWD / "annotations"

if not (ANNOTATIONS_DIR / "reva_labels.tsv").exists():
    raise FileNotFoundError(f"Could not find annotation files in {ANNOTATIONS_DIR}")

ANNOTATIONS_DIR

In [ ]:
def parse_labels(value) -> frozenset:
    """Parse comma-separated label string to frozenset of stripped labels.
    NaN, empty string, and whitespace-only all return frozenset({'_unknown'})."""
    if pd.isna(value) or str(value).strip() == "":
        return frozenset({'_unknown'})
    return frozenset(lbl.strip() for lbl in str(value).split(",") if lbl.strip())


def clean_merged(label_set: frozenset) -> frozenset:
    """Remove _unknown from sets that also contain substantive labels.
    Keep _unknown only when it is the sole label."""
    real = label_set - {'_unknown'}
    return real if real else frozenset({'_unknown'})


def serialize(label_set: frozenset) -> str:
    """Sort labels alphabetically and join with ', '."""
    return ", ".join(sorted(label_set))


def explode_counts(series_of_sets: pd.Series) -> pd.Series:
    """Count label frequency across items (multi-label explode)."""
    exploded = pd.Series([label for s in series_of_sets for label in s])
    return exploded.value_counts().sort_values(ascending=False)


def merge_task(
    reva_path: Path,
    ryan_path: Path,
    join_keys: list[str],
    label_col: str,
    merged_col: str,
    out_path: Path,
):
    reva = pd.read_csv(reva_path, sep="\t", dtype=str)
    ryan = pd.read_csv(ryan_path, sep="\t", dtype=str)

    merged = reva.merge(
        ryan[[*join_keys, label_col]],
        on=join_keys,
        how="inner",
        suffixes=("_reva_orig", "_ryan_orig"),
    )

    # Reconstruct the reva schema column name because merge suffixes rename overlaps.
    merged[label_col] = merged[f"{label_col}_reva_orig"]

    # Preserve original strings in explicit columns.
    merged[f"{label_col}_reva"] = merged[f"{label_col}_reva_orig"]
    merged[f"{label_col}_ryan"] = merged[f"{label_col}_ryan_orig"]

    reva_sets = merged[f"{label_col}_reva"].apply(parse_labels)
    ryan_sets = merged[f"{label_col}_ryan"].apply(parse_labels)

    merged_raw = [a | b for a, b in zip(reva_sets, ryan_sets)]
    merged_clean = [clean_merged(s) for s in merged_raw]

    merged[merged_col] = [serialize(s) for s in merged_clean]

    # Output file: all columns from reva file + required merge columns.
    base_cols = list(reva.columns)
    out_cols = [*base_cols, f"{label_col}_reva", f"{label_col}_ryan", merged_col]
    merged_out = merged[out_cols].copy()
    merged_out.to_csv(out_path, sep="\t", index=False)

    both_unknown = sum(
        (a == frozenset({'_unknown'}) and b == frozenset({'_unknown'}))
        for a, b in zip(reva_sets, ryan_sets)
    )
    unknown_stripped = sum(
        ('_unknown' in raw and cleaned != raw)
        for raw, cleaned in zip(merged_raw, merged_clean)
    )
    unique_labels = len(set().union(*merged_clean)) if merged_clean else 0

    stats = {
        'total_items': int(len(merged_out)),
        'both_unknown': int(both_unknown),
        'unknown_stripped': int(unknown_stripped),
        'unique_labels': int(unique_labels),
    }

    counts = pd.DataFrame({
        'Reva': explode_counts(reva_sets),
        'Ryan': explode_counts(ryan_sets),
        'Merged': explode_counts(pd.Series(merged_clean)),
    }).fillna(0).astype(int).sort_values('Merged', ascending=False)

    diagnostics = pd.DataFrame([
        {'category': 'both_unknown', 'count': int(both_unknown)},
        {'category': 'unknown_stripped', 'count': int(unknown_stripped)},
        {
            'category': 'unknown_retained_only',
            'count': int(sum(s == frozenset({'_unknown'}) for s in merged_clean)),
        },
    ])

    return merged_out, stats, counts, diagnostics

In [ ]:
labels_merged, labels_stats, labels_counts, labels_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_labels.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_labels.tsv",
    join_keys=["dataset", "raw_target_label"],
    label_col="dest_label",
    merged_col="dest_label_merged",
    out_path=ANNOTATIONS_DIR / "merged_labels.tsv",
)

glossary_merged, glossary_stats, glossary_counts, glossary_diag = merge_task(
    reva_path=ANNOTATIONS_DIR / "reva_glossary.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_glossary.tsv",
    join_keys=["term"],
    label_col="inferred_target",
    merged_col="inferred_target_merged",
    out_path=ANNOTATIONS_DIR / "merged_glossary.tsv",
)

print("Wrote:")
print(ANNOTATIONS_DIR / "merged_labels.tsv")
print(ANNOTATIONS_DIR / "merged_glossary.tsv")

In [ ]:
# Join source_domain and tier from the provenance-classified glossary into merged_glossary.tsv.
# glossary.tsv is written by 00_glossary_formatter.ipynb and already carries these columns.
_provenance = pd.read_csv(
    ANNOTATIONS_DIR.parent / 'outputs' / 'glossary' / 'glossary.tsv',
    sep='\t',
    usecols=['term', 'source_domain', 'tier'],
)
glossary_merged = glossary_merged.merge(_provenance, on='term', how='left')
glossary_merged.to_csv(ANNOTATIONS_DIR / 'merged_glossary.tsv', sep='\t', index=False)

tier_coverage = glossary_merged['tier'].notna().sum()
print(f'Re-wrote merged_glossary.tsv with tier ({tier_coverage}/{len(glossary_merged)} rows matched)')
glossary_merged[['term', 'source_domain', 'tier', 'inferred_target_merged']].head(8)

In [ ]:
print("Labels task")
print("-----------")
print(f"Total items:                 {labels_stats['total_items']}")
print(f"Both _unknown after merge:   {labels_stats['both_unknown']}")
print(f"_unknown stripped from union: {labels_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {labels_stats['unique_labels']}")
print()
print("Glossary task")
print("-------------")
print(f"Total items:                 {glossary_stats['total_items']}")
print(f"Both _unknown after merge:   {glossary_stats['both_unknown']}")
print(f"_unknown stripped from union: {glossary_stats['unknown_stripped']}")
print(f"Unique labels in merged:     {glossary_stats['unique_labels']}")

In [ ]:
def plot_top_label_breakdown(counts_df: pd.DataFrame, title: str, top_n: int = 20) -> None:
    top = counts_df.head(top_n).copy()

    ax = top[["Reva", "Ryan", "Merged"]].plot(
        kind="bar",
        figsize=(14, 6),
        width=0.85,
    )
    ax.set_title(title)
    ax.set_xlabel("Label")
    ax.set_ylabel("Count across items")
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_unknown_diagnostics(diag_df: pd.DataFrame, title: str) -> None:
    ax = diag_df.plot(kind="bar", x="category", y="count", legend=False, figsize=(8, 4), color="#5a88c8")
    ax.set_title(title)
    ax.set_xlabel("Diagnostic category")
    ax.set_ylabel("Count")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## Visual Breakdown: Labels Task

In [ ]:
plot_top_label_breakdown(labels_counts, "Labels Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(labels_diag, "Labels Task: _unknown Handling Diagnostics")
labels_counts.head(20)

## Visual Breakdown: Glossary Task

In [ ]:
plot_top_label_breakdown(glossary_counts, "Glossary Task: Top Labels by Merged Count", top_n=20)
plot_unknown_diagnostics(glossary_diag, "Glossary Task: _unknown Handling Diagnostics")
glossary_counts.head(20)

---

## Disagreement Analysis

The union-merge strategy resolves disagreements by taking the union of label sets. This is maximally inclusive but may be inappropriate when annotators assign *incompatible* labels rather than complementary ones.

Two questions motivate re-examining the naive scheme:

1. **Are most disagreements additive?**  
   An *additive* disagreement means one annotator's real-label set is a (strict) subset of the other's — the union just adds the extra labels without any conflict.  One annotator is empty (assigned `_unknown` or nothing) while the other has labels also counts as additive.

2. **How many cases are completely non-overlapping (disjoint)?**  
   A *disjoint* disagreement means the two annotators share zero labels — the union silently merges two incompatible judgements, with no signal about which (if either) is correct.

Categories:
| Class | Meaning |
|---|---|
| `exact_agreement` | Both annotators assigned identical real labels |
| `both_unknown` | Neither assigned any real labels |
| `additive` | Real labels differ; one set ⊆ the other (union is benign) |
| `partial_overlap` | Real labels differ; non-empty intersection; neither is a subset |
| `disjoint` | Real labels differ; no shared labels at all |

In [ ]:
_ORDER = ['exact_agreement', 'both_unknown', 'additive', 'partial_overlap', 'disjoint']
_COLORS = {
    'exact_agreement': '#2ca02c',
    'both_unknown':    '#aec7e8',
    'additive':        '#1f77b4',
    'partial_overlap': '#ff7f0e',
    'disjoint':        '#d62728',
}
_DISPLAY = {
    'exact_agreement': 'Exact agreement',
    'both_unknown':    'Both unknown',
    'additive':        'Additive (one ⊆ other)',
    'partial_overlap': 'Partial overlap',
    'disjoint':        'Disjoint (no shared labels)',
}


def classify_disagreement(reva_val, ryan_val) -> str:
    """
    Classify the relationship between two annotators' label sets.

    Strips _unknown placeholders before testing subset/overlap relationships
    so that 'I don't know' never counts as a substantive label assignment.
    """
    r = parse_labels(reva_val) - {'_unknown'}
    y = parse_labels(ryan_val) - {'_unknown'}

    if not r and not y:
        return 'both_unknown'
    if r == y:
        return 'exact_agreement'
    # One annotator left no real labels — the union is benign
    if not r or not y:
        return 'additive'
    ix = r & y
    if not ix:
        return 'disjoint'
    if r <= y or y <= r:
        return 'additive'
    return 'partial_overlap'


def disagreement_summary(df: pd.DataFrame, reva_col: str, ryan_col: str):
    classes = df.apply(
        lambda row: classify_disagreement(row[reva_col], row[ryan_col]), axis=1
    )
    counts = classes.value_counts().reindex(_ORDER, fill_value=0)
    total = len(df)
    disagree_n = int(total - counts['exact_agreement'] - counts['both_unknown'])
    summary = pd.DataFrame({
        'count': counts,
        'pct_total': (counts / total * 100).round(1),
    })
    return summary, disagree_n, classes


labels_dis_summary, labels_dis_n, _   = disagreement_summary(
    labels_merged, 'dest_label_reva', 'dest_label_ryan'
)
glossary_dis_summary, glossary_dis_n, _ = disagreement_summary(
    glossary_merged, 'inferred_target_reva', 'inferred_target_ryan'
)

In [ ]:
def _print_disagreement_report(summary: pd.DataFrame, task_name: str, disagree_n: int) -> None:
    total = summary['count'].sum()
    print(f"{task_name}  (n={total})")
    print("-" * 56)
    for cls in _ORDER:
        cnt = summary.loc[cls, 'count']
        pct = summary.loc[cls, 'pct_total']
        print(f"  {_DISPLAY[cls]:<30s}: {cnt:>4d}  ({pct:>5.1f}%)")
    print()
    if disagree_n > 0:
        print(f"  Among {disagree_n} true disagreements (excl. exact_agreement + both_unknown):")
        for cls in ['additive', 'partial_overlap', 'disjoint']:
            cnt = summary.loc[cls, 'count']
            pct = cnt / disagree_n * 100
            print(f"    {_DISPLAY[cls]:<30s}: {cnt:>4d}  ({pct:>5.1f}%)")
    print()


_print_disagreement_report(labels_dis_summary, "Labels task", labels_dis_n)
_print_disagreement_report(glossary_dis_summary, "Glossary task", glossary_dis_n)

In [ ]:
# Overview: all five categories side-by-side for both tasks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, summary, title in [
    (axes[0], labels_dis_summary,  f'Labels task  (n={labels_dis_summary["count"].sum()})'),
    (axes[1], glossary_dis_summary, f'Glossary task  (n={glossary_dis_summary["count"].sum()})'),
]:
    disp_labels = [_DISPLAY[c] for c in _ORDER]
    counts_vals = [summary.loc[c, 'count'] for c in _ORDER]
    colors_vals = [_COLORS[c] for c in _ORDER]
    bars = ax.barh(disp_labels, counts_vals, color=colors_vals)
    for bar, val in zip(bars, counts_vals):
        if val > 0:
            ax.text(
                bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9,
            )
    ax.set_title(title)
    ax.set_xlabel('Number of items')
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Full agreement-class breakdown by task', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom in: structure of true disagreements only (excludes exact_agreement + both_unknown)
_DIS_CATS = ['additive', 'partial_overlap', 'disjoint']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, summary, disagree_n, title in [
    (axes[0], labels_dis_summary,   labels_dis_n,
     f'Labels task\n({labels_dis_n} true disagreements)'),
    (axes[1], glossary_dis_summary, glossary_dis_n,
     f'Glossary task\n({glossary_dis_n} true disagreements)'),
]:
    if disagree_n == 0:
        ax.text(0.5, 0.5, 'No disagreements', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.axis('off')
        continue

    vals = [summary.loc[c, 'count'] for c in _DIS_CATS]
    colors_sub = [_COLORS[c] for c in _DIS_CATS]
    display_sub = [_DISPLAY[c] for c in _DIS_CATS]

    wedges, texts, autotexts = ax.pie(
        vals,
        labels=display_sub,
        colors=colors_sub,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.72,
        wedgeprops={'linewidth': 0.6, 'edgecolor': 'white'},
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title(title)

plt.suptitle('Structure of disagreements\n(exact_agreement and both_unknown excluded)', fontsize=11)
plt.tight_layout()
plt.show()

### Disagreement case tables

Full listing of every item where Reva and Ryan disagreed, with its disagreement class.  
`both_unknown` and `exact_agreement` rows are excluded — only the 201 (labels) and 97 (glossary) true disagreements are shown.

Sorted by class so the most problematic cases (`disjoint`) are grouped together.

In [ ]:
_DIS_ORDER = {'disjoint': 0, 'partial_overlap': 1, 'additive': 2}

# --- Labels task ---
labels_merged['disagreement_class'] = labels_merged.apply(
    lambda row: classify_disagreement(row['dest_label_reva'], row['dest_label_ryan']), axis=1
)
labels_disagreements = (
    labels_merged[labels_merged['disagreement_class'].isin(_DIS_ORDER)]
    .assign(_sort=lambda df: df['disagreement_class'].map(_DIS_ORDER))
    .sort_values(['_sort', 'dataset', 'raw_target_label'])
    .drop(columns=['_sort'])
    [['dataset', 'raw_target_label', 'dest_label_reva', 'dest_label_ryan', 'disagreement_class']]
    .reset_index(drop=True)
)

print(f"Labels task — {len(labels_disagreements)} true disagreements "
      f"({labels_disagreements['disagreement_class'].value_counts().to_dict()})")
display(labels_disagreements)

In [ ]:
# --- Glossary task ---
glossary_merged['disagreement_class'] = glossary_merged.apply(
    lambda row: classify_disagreement(row['inferred_target_reva'], row['inferred_target_ryan']), axis=1
)
glossary_disagreements = (
    glossary_merged[glossary_merged['disagreement_class'].isin(_DIS_ORDER)]
    .assign(_sort=lambda df: df['disagreement_class'].map(_DIS_ORDER))
    .sort_values(['_sort', 'term'])
    .drop(columns=['_sort'])
    [['term', 'inferred_target_reva', 'inferred_target_ryan', 'tier', 'disagreement_class']]
    .reset_index(drop=True)
)

print(f"Glossary task — {len(glossary_disagreements)} true disagreements "
      f"({glossary_disagreements['disagreement_class'].value_counts().to_dict()})")
display(glossary_disagreements)